# Loan Applications: Cleaning and Exploratory Analysis

**Author:** Maria Pupysheva · **Date:** October 2026

**Data:** Loan Prediction dataset (Kaggle): 614 home loan applications with applicant details,
loan details and the lender's decision (`Loan_Status`). Source: [Kaggle](https://www.kaggle.com/datasets/burak3ergun/loan-data-set)

**Goal:** clean the data, engineer affordability features (such as repayment-to-income) and
explore which factors are linked to loan approval. This is practice for the data-preparation
part of the Westpac Hackathon project.

**Note:** `Loan_Status` records whether the loan was **approved**, not whether it was repaid.
Findings describe the lender's past decisions, not repayment risk.

**Tools:** Python 3.12 · pandas · NumPy · Matplotlib

**Contents**
1. Setup · 2. Data overview · 3. Cleaning · 4. Feature engineering · 5. Exploratory analysis · 6. Summary

## 1. Setup
Load the raw CSV. The data file is kept in `data/` and is not uploaded to GitHub.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 20)   # show all columns in wide tables

In [2]:
loans_raw = pd.read_csv("../data/loan_data_set.csv")
loans_raw.shape

(614, 13)

## 2. Data overview
A first look at the structure, types, missing values and distributions before any cleaning.

### Q1. Structure
How many applications and columns are there, and what does one row represent?

In [3]:
print(f"Applications: {len(loans_raw)}, columns: {loans_raw.shape[1]}")
loans_raw.head()

Applications: 614, columns: 13


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


**Observation:** 614 loan applications with 13 columns: one row per application, with applicant
details (gender, marital status, dependents, education, employment, income), loan details
(amount, term), credit history, property area and the lender's decision (`Loan_Status`).

### Q2. Column types
Which columns are numbers and which are text, and is any column stored as the wrong type?

In [4]:
loans_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    str    
 1   Gender             601 non-null    str    
 2   Married            611 non-null    str    
 3   Dependents         599 non-null    str    
 4   Education          614 non-null    str    
 5   Self_Employed      582 non-null    str    
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    str    
 12  Loan_Status        614 non-null    str    
dtypes: float64(4), int64(1), str(8)
memory usage: 62.5 KB


In [5]:
display(loans_raw["Dependents"].unique())
display(loans_raw["Credit_History"].unique())

<StringArray>
['0', '1', '2', '3+', nan]
Length: 5, dtype: str

array([ 1.,  0., nan])

**Observation:**
- `Dependents` is text because of the value `3+`; it needs converting to a number
  (treating `3+` as 3, meaning "3 or more").
- `Credit_History` is stored as float only because it has missing values. Its values are
  1 (meets the lender's credit guidelines) and 0 (does not), so it is really a yes/no flag.

### Q3. Missing values
Which columns have missing values, how many, and what share of rows?

In [6]:
missing = (
    loans_raw.isna().sum()
    .to_frame("missing")
    .assign(missing_pct=lambda df: df["missing"] / len(loans_raw))
    .query("missing > 0")
    .sort_values("missing", ascending=False)
)
missing.style.format({"missing_pct": "{:.1%}"})

,missing,missing_pct
Credit_History,50,8.1%
Self_Employed,32,5.2%
LoanAmount,22,3.6%
Dependents,15,2.4%
Loan_Amount_Term,14,2.3%
Gender,13,2.1%
Married,3,0.5%


**Observation:** 7 columns have missing values, from 0.5% (`Married`) to 8.1% (`Credit_History`).
`Credit_History` matters most: it is the most lending-relevant column and has the most missing
values (50), so how it is handled can change the conclusions. `Self_Employed` is next (32, 5.2%).

### Q4. Numeric columns
Distribution of incomes, loan amount and loan term.

In [7]:
numeric_cols = ["ApplicantIncome", "CoapplicantIncome", "LoanAmount", "Loan_Amount_Term"]
loans_raw[numeric_cols].describe().T.round(1)

,count,mean,std,min,25%,50%,75%,max
ApplicantIncome,614.0,5403.5,6109.0,150.0,2877.5,3812.5,5795.0,81000.0
CoapplicantIncome,614.0,1621.2,2926.2,0.0,0.0,1188.5,2297.2,41667.0
LoanAmount,592.0,146.4,85.6,9.0,100.0,128.0,168.0,700.0
Loan_Amount_Term,600.0,342.0,65.1,12.0,360.0,360.0,360.0,480.0


In [8]:
loans_raw["Loan_Amount_Term"].value_counts(dropna=False).sort_index()

Loan_Amount_Term
12.0       1
36.0       2
60.0       2
84.0       4
120.0      3
180.0     44
240.0      4
300.0     13
360.0    512
480.0     15
NaN       14
Name: count, dtype: int64

**Observation:**
- Incomes are strongly right-skewed: applicant income has a median of 3,812 but a maximum of
  81,000; the mean (5,403) is pulled up by a few very high earners, so medians are more
  representative.
- At least 25% of applicants have no co-applicant (co-applicant income of 0).
- `LoanAmount` is most likely in thousands (median 128, minimum 9); incomes are most likely
  monthly. The dataset does not document units, so these are assumptions.
- `Loan_Amount_Term` is in months: 360 months (30 years) is by far the most common (512 of 600),
  with a few unusual short terms down to 12 months.

### Q5. Categorical columns
Values and counts of each text column, including missing values.

In [9]:
categorical_cols = ["Gender", "Married", "Dependents", "Education", "Self_Employed", "Property_Area"]

for col in categorical_cols:
    display(loans_raw[col].value_counts(dropna=False).rename_axis(col).reset_index(name="applications"))

,Gender,applications
0,Male,489
1,Female,112
2,NaN,13


,Married,applications
0,Yes,398
1,No,213
2,NaN,3


,Dependents,applications
0,0,345
1,1,102
2,2,101
3,3+,51
4,NaN,15


,Education,applications
0,Graduate,480
1,Not Graduate,134


,Self_Employed,applications
0,No,500
1,Yes,82
2,NaN,32


,Property_Area,applications
0,Semiurban,233
1,Urban,202
2,Rural,179


In [10]:
print(f"Loan_ID is unique: {loans_raw['Loan_ID'].is_unique}")

Loan_ID is unique: True


**Observation:** Categories are spelled consistently and `Loan_ID` is unique (no duplicates).
Groups are unbalanced: 80% of known applicants are male, 65% are married and only 14% are
self-employed, so approval rates for small groups will be less reliable. `Dependents` should be
treated as a number (0, 1, 2, 3+), not as free text.

### Q6. The target
What share of applications was approved?

In [11]:
status = (
    loans_raw["Loan_Status"]
    .value_counts()
    .rename_axis("loan_status")
    .reset_index(name="applications")
)
status["share"] = status["applications"] / status["applications"].sum()
status.style.format({"share": "{:.1%}"}).hide(axis="index")

loan_status,applications,share
Y,422,68.7%
N,192,31.3%


**Observation:** 68.7% of applications were approved and 31.3% rejected. The classes are
unbalanced: a rule that approves everyone would already be right 68.7% of the time, so any
model must beat that baseline, and groups should be compared by approval rate, not by counts.

### Data quality notes
These problems form the cleaning plan for section 3.

1. Missing values in 7 columns, most in `Credit_History` (8.1%) and `Self_Employed` (5.2%).
2. `Dependents` is text (`3+`); convert to a number.
3. `Credit_History` is a 0/1 flag stored as float; convert after handling missing values.
4. Incomes are heavily right-skewed with extreme values (up to 81,000): check for outliers.
5. Units are undocumented: `LoanAmount` assumed in thousands, incomes assumed monthly.
6. A few unusual loan terms (12–84 months) to review.

## 3. Cleaning
Each step fixes one problem from the data quality notes. The raw data stays untouched in
`loans_raw`; all changes are made on a copy, `loans`.

### Q7. Working copy
All cleaning is done on a copy, so the raw data stays untouched.

In [12]:
loans = loans_raw.copy()

# Test: change one cell in the copy and check the raw data is unaffected
loans.loc[0, "Gender"] = "TEST"
print(f"Copy:     {loans.loc[0, 'Gender']}")
print(f"Raw data: {loans_raw.loc[0, 'Gender']}")

loans = loans_raw.copy()   # recreate a clean copy after the test

Copy:     TEST
Raw data: Male


**Observation:** All cleaning is done on `loans`, a copy of the raw data. Changing the copy
does not affect `loans_raw`, so every cleaning step can be checked against, and rerun from,
the original data.

### Q8. `Dependents`: from text to a number

In [13]:
loans["Dependents"] = pd.to_numeric(loans["Dependents"].replace("3+", "3")).astype("Int64")

display(loans["Dependents"].value_counts(dropna=False).sort_index())
print(f"Type: {loans['Dependents'].dtype}")

Dependents
0       345
1       102
2       101
3        51
<NA>     15
Name: count, dtype: Int64

Type: Int64


**Observation:** `Dependents` is now a whole number (nullable `Int64`): 0, 1, 2 or 3, where
**3 means "3 or more"** (the original `3+`). The 15 missing values are kept as missing for now
and handled in the missing-values step.

### Q9. `Credit_History`: what to do with the 50 missing values?
Before deciding, compare the approval rate of applicants with a good, bad and missing credit history.

In [14]:
# Temporary label for each credit history group, including missing
credit_label = (
    loans["Credit_History"]
    .map({1: "Good", 0: "Bad"})
    .fillna("Missing")
)

credit_vs_approval = (
    loans
    .assign(credit_group=credit_label, approved=loans["Loan_Status"] == "Y")
    .groupby("credit_group")
    .agg(applications=("Loan_ID", "count"), approval_rate=("approved", "mean"))
    .reset_index()
)
credit_vs_approval.style.format({"approval_rate": "{:.1%}"}).hide(axis="index")

credit_group,applications,approval_rate
Bad,89,7.9%
Good,475,79.6%
Missing,50,74.0%


**Observation:** Credit history is strongly linked to approval: 79.6% of applicants with a good
credit history were approved, compared with only 7.9% with a bad one (about 10 times lower).
Applicants with a missing credit history were approved at 74.0%, much closer to the "good"
group than the "bad" one, so a missing value does not mean a bad history.

**Decision:** filling missing values with 0 (bad) would contradict the data, and filling with 1
(good) would invent information. The missing values are therefore kept as a separate category,
`Missing`, in a new text column `credit_history` (Good / Bad / Missing). The original
`Credit_History` column is kept unchanged.

In [15]:
loans["credit_history"] = (
    loans["Credit_History"]
    .map({1: "Good", 0: "Bad"})
    .fillna("Missing")
)
loans["credit_history"].value_counts()

credit_history
Good       475
Bad         89
Missing     50
Name: count, dtype: int64

### Q10. The remaining missing values
`Self_Employed` is checked first: does the missing group behave differently from the others?

In [16]:
selfempl_vs_approval = (
    loans
    .assign(approved=loans["Loan_Status"] == "Y")
    .groupby("Self_Employed", dropna=False)
    .agg(applications=("Loan_ID", "count"), approval_rate=("approved", "mean"))
    .reset_index()
)
selfempl_vs_approval.style.format({"approval_rate": "{:.1%}"}).hide(axis="index")

Self_Employed,applications,approval_rate
No,500,68.6%
Yes,82,68.3%
nan,32,71.9%


**Observation:** Employment type is barely linked to approval: employed 68.6%, self-employed
68.3%, missing 71.9%. The missing group is small (32) and its rate is close to the others,
so filling it with the most common value ("No") does not distort any conclusion.

**Decisions for the remaining missing values:**

| Column | Missing | Filled with | Why |
|---|---|---|---|
| Self_Employed | 32 | "No" (most common) | the missing group's approval rate matches the others (above) |
| LoanAmount | 22 | median | right-skewed with outliers up to 700; the median is not pulled up by them |
| Dependents | 15 | 0 (most common) | 0 dependents is by far the most common (345 of 599) |
| Loan_Amount_Term | 14 | 360 (most common) | 85% of loans are 360 months |
| Gender | 13 | "Unknown" | not needed for affordability, and guessing gender would invent information |
| Married | 3 | "Yes" (most common) | only 3 rows, so the method makes no practical difference |

In [17]:
fill_values = {
    "Self_Employed":    loans["Self_Employed"].mode()[0],     # "No"
    "LoanAmount":       loans["LoanAmount"].median(),
    "Dependents":       loans["Dependents"].mode()[0],        # 0
    "Loan_Amount_Term": loans["Loan_Amount_Term"].mode()[0],  # 360
    "Gender":           "Unknown",
    "Married":          loans["Married"].mode()[0],           # "Yes"
}
loans = loans.fillna(fill_values)

# Check: only the original Credit_History should still have missing values
loans.isna().sum()[lambda s: s > 0]

Credit_History    50
dtype: int64

**Observation:** All missing values are now handled. The only remaining gaps are the 50 in the
original `Credit_History` column, kept on purpose: the analysis uses the new `credit_history`
column, where they form the "Missing" category (see Q9).

### Q11. Outliers in applicant income
IQR rule: values above Q3 + 1.5 × IQR or below Q1 − 1.5 × IQR.

In [18]:
q1 = loans["ApplicantIncome"].quantile(0.25)
q3 = loans["ApplicantIncome"].quantile(0.75)
income_iqr = q3 - q1

upper = q3 + 1.5 * income_iqr
lower = q1 - 1.5 * income_iqr
print(f"Lower threshold: {lower:,.0f}   Upper threshold: {upper:,.0f}")

outliers = loans[(loans["ApplicantIncome"] < lower) | (loans["ApplicantIncome"] > upper)]
print(f"Outliers: {len(outliers)} of {len(loans)} applicants ({len(outliers) / len(loans):.1%})")
print(f"Approval rate of outliers: {(outliers['Loan_Status'] == 'Y').mean():.1%}")

outliers[["ApplicantIncome", "CoapplicantIncome", "LoanAmount", "Education",
          "Property_Area", "credit_history", "Loan_Status"]].sort_values("ApplicantIncome", ascending=False)

Lower threshold: -1,499   Upper threshold: 10,171
Outliers: 50 of 614 applicants (8.1%)
Approval rate of outliers: 66.0%


,ApplicantIncome,CoapplicantIncome,LoanAmount,Education,Property_Area,credit_history,Loan_Status
409,81000,0.0,360.0,Graduate,Rural,Bad,N
333,63337,0.0,490.0,Graduate,Urban,Good,Y
171,51763,0.0,700.0,Graduate,Urban,Good,Y
155,39999,0.0,600.0,Graduate,Semiurban,Bad,Y
185,39147,4750.0,120.0,Graduate,Semiurban,Good,Y
443,37719,0.0,152.0,Graduate,Semiurban,Good,Y
183,33846,0.0,260.0,Graduate,Semiurban,Good,N
126,23803,0.0,370.0,Graduate,Rural,Good,Y
506,20833,6667.0,480.0,Graduate,Urban,Missing,Y
284,20667,0.0,128.0,Graduate,Rural,Good,N


**Observation:** By the IQR rule, 50 applicants (8.1%) have unusually high incomes, from
10,408 up to 81,000; there are no low outliers, because the lower threshold is negative
(-1,499) and income cannot be below zero. These values look genuine rather than errors: the
applicants are almost all graduates (48 of 50), mostly without a co-applicant, and they request
the largest loans in the data (up to 700). Their approval rate (66%) is close to the overall
68.7%, so high income alone does not raise the chance of approval. Credit history matters more:
the highest earner (81,000) was rejected with a bad credit history, as were most other
high earners with a bad history.

**Decision:** the outliers are kept. Medians are used to describe income, and a log transform
or capping can be considered later for modelling.

## 4. Feature engineering
New columns that describe affordability: how much of an applicant's income the loan would take.

### Q12. Total income
Lenders assess the household's ability to repay, so applicant and co-applicant income are combined.

In [19]:
loans["total_income"] = loans["ApplicantIncome"] + loans["CoapplicantIncome"]

print(f"Median applicant income: {loans['ApplicantIncome'].median():,.0f}")
print(f"Median total income:     {loans['total_income'].median():,.0f}")

Median applicant income: 3,812
Median total income:     5,416


**Observation:** `total_income` combines applicant and co-applicant income. Its median (5,417) is
about 42% higher than the median applicant income alone (3,813), so many households have a
second income that matters for affordability. Total income is used for the affordability
features below.

### Q13. Estimated monthly repayment
Standard loan repayment formula at an assumed interest rate, checked on a known case first.

In [20]:
annual_rate = 0.09                        # assumption: 9% per year (the dataset has no rate)
r = annual_rate / 12                      # monthly interest rate

def repayment(principal, months):
    """Monthly payment of a standard loan repaid in equal instalments."""
    return principal * r * (1 + r) ** months / ((1 + r) ** months - 1)

# Check on a known case: 100,000 over 30 years at 9% ≈ 805 per month
print(f"Test: 100,000 over 360 months at 9% → {repayment(100_000, 360):,.0f} per month")

loans["monthly_repayment"] = repayment(loans["LoanAmount"] * 1000, loans["Loan_Amount_Term"])  # LoanAmount is in thousands
loans[["LoanAmount", "Loan_Amount_Term", "monthly_repayment"]].head().round(0)

Test: 100,000 over 360 months at 9% → 805 per month


,LoanAmount,Loan_Amount_Term,monthly_repayment
0,128.0,360.0,1030.0
1,128.0,360.0,1030.0
2,66.0,360.0,531.0
3,120.0,360.0,966.0
4,141.0,360.0,1135.0


**Observation:** `monthly_repayment` estimates each applicant's monthly payment with the standard
loan repayment formula, using an **assumed interest rate of 9% per year** (the dataset has no
rate). For example, a 128,000 loan over 360 months costs about 1,030 per month. The formula was
checked on a known case (100,000 over 30 years at 9% ≈ 805 per month). Because the rate is a
variable, it can be changed to test other scenarios.

### Q14. Repayment-to-income ratio
The share of monthly household income that would go to the loan.

In [21]:
loans["repayment_to_income"] = loans["monthly_repayment"] / loans["total_income"]

print(f"Median repayment-to-income: {loans['repayment_to_income'].median():.1%}")
loans["repayment_to_income"].describe().to_frame().T.style.format(
    {"count": "{:.0f}", "mean": "{:.1%}", "std": "{:.1%}", "min": "{:.1%}", "25%": "{:.1%}",
     "50%": "{:.1%}", "75%": "{:.1%}", "max": "{:.1%}"}
)

Median repayment-to-income: 19.8%


,count,mean,std,min,25%,50%,75%,max
repayment_to_income,614,20.2%,10.1%,2.0%,15.8%,19.8%,23.3%,141.6%


In [22]:
high_ratio = loans[loans["repayment_to_income"] > 0.5]
print(f"Applicants above 50%: {len(high_ratio)}")

high_ratio[["total_income", "LoanAmount", "Loan_Amount_Term", "monthly_repayment",
            "repayment_to_income", "credit_history", "Loan_Status"]] \
    .sort_values("repayment_to_income", ascending=False) \
    .style.format({"total_income": "{:,.0f}", "monthly_repayment": "{:,.0f}", "repayment_to_income": "{:.0%}"})

Applicants above 50%: 7


,total_income,LoanAmount,Loan_Amount_Term,monthly_repayment,repayment_to_income,credit_history,Loan_Status
262,"3,481",155.000000,36.000000,"4,929",142%,Good,N
497,"7,482",111.000000,12.000000,"9,707",130%,Missing,Y
546,"3,358",80.000000,36.000000,"2,544",76%,Good,N
255,"3,083",255.000000,360.000000,"2,052",67%,Good,Y
338,"1,830",128.000000,360.000000,"1,030",56%,Bad,N
216,"1,950",135.000000,360.000000,"1,086",56%,Good,N
305,"2,000",128.000000,360.000000,"1,030",51%,Good,N


In [23]:
# Were the loan amounts of rows 338 and 305 originally missing (filled with the median)?
loans_raw.loc[[338, 305], "LoanAmount"]

338   NaN
305   NaN
Name: LoanAmount, dtype: float64

**Observation:** The median repayment-to-income ratio is 19.8%, a realistic share of monthly
income, which supports the assumption that incomes are monthly. Only 7 applicants exceed 50%,
and 5 of them were rejected (71%, vs 31% overall), so a high ratio goes with rejection.
The three highest ratios (76–142%) come from very short terms (12–36 months): the standard
repayment formula probably does not fit these loans (the term may be recorded differently, or
they are repaid in a lump sum), which is a limitation of this feature for short-term loans.
Two of the high-ratio applicants (rows 338 and 305) originally had no loan amount; it was filled
with the median (128), so their ratio is partly an estimate.

### Q15. Does affordability affect approval?
Applicants are grouped into affordability bands with fixed thresholds.

In [24]:
bins   = [0, 0.15, 0.25, 0.40, float("inf")]
labels = ["Low (<15%)", "Medium (15–25%)", "High (25–40%)", "Very high (>40%)"]
loans["affordability"] = pd.cut(loans["repayment_to_income"], bins=bins, labels=labels)

afford_vs_approval = (
    loans
    .assign(approved=loans["Loan_Status"] == "Y")
    .groupby("affordability", observed=False)
    .agg(applications=("Loan_ID", "count"), approval_rate=("approved", "mean"))
    .reset_index()
)
print(f"Applicants in bands: {afford_vs_approval['applications'].sum()} of {len(loans)}")
afford_vs_approval.style.format({"approval_rate": "{:.1%}"}).hide(axis="index")

Applicants in bands: 614 of 614


affordability,applications,approval_rate
Low (<15%),125,70.4%
Medium (15–25%),394,70.3%
High (25–40%),82,63.4%
Very high (>40%),13,38.5%


**Observation:** Affordability matters only at the extreme end. Applicants whose repayment takes
up to 25% of income are approved at the same rate (≈70%), those at 25–40% slightly less (63.4%),
and those above 40% at only 38.5%, about half the usual rate. The "very high" group is small
(13 applicants), so its rate is less certain. The effect is much weaker than credit history
(79.6% vs 7.9% approval), so credit history drove this lender's decisions far more than
affordability did.

**Note:** the 9% interest rate is an assumption; a higher rate would move more applicants into
the higher bands. This is the basis for the stress-testing idea in the hackathon project.

## 5. Exploratory analysis
Which applicant and loan characteristics are linked to approval?

### Q16. Approval rate by every characteristic
One function, applied to each column, then columns ranked by the gap between their highest and lowest approval rate.

In [25]:
def approval_by(column):
    """Number of applications and approval rate for each value of a column."""
    return (
        loans
        .assign(approved=loans["Loan_Status"] == "Y")
        .groupby(column, dropna=False, observed=False)
        .agg(applications=("Loan_ID", "count"), approval_rate=("approved", "mean"))
        .reset_index()
    )

columns = ["Gender", "Married", "Dependents", "Education", "Self_Employed",
           "Property_Area", "credit_history", "affordability"]

for column in columns:
    display(approval_by(column).style.format({"approval_rate": "{:.1%}"}).hide(axis="index"))

Gender,applications,approval_rate
Female,112,67.0%
Male,489,69.3%
Unknown,13,61.5%


Married,applications,approval_rate
No,213,62.9%
Yes,401,71.8%


Dependents,applications,approval_rate
0,360,68.6%
1,102,64.7%
2,101,75.2%
3,51,64.7%


Education,applications,approval_rate
Graduate,480,70.8%
Not Graduate,134,61.2%


Self_Employed,applications,approval_rate
No,532,68.8%
Yes,82,68.3%


Property_Area,applications,approval_rate
Rural,179,61.5%
Semiurban,233,76.8%
Urban,202,65.8%


credit_history,applications,approval_rate
Bad,89,7.9%
Good,475,79.6%
Missing,50,74.0%


affordability,applications,approval_rate
Low (<15%),125,70.4%
Medium (15–25%),394,70.3%
High (25–40%),82,63.4%
Very high (>40%),13,38.5%


In [26]:
gaps = pd.DataFrame(
    {
        "column": columns,
        "gap": [approval_by(c)["approval_rate"].max() - approval_by(c)["approval_rate"].min() for c in columns],
    }
).sort_values("gap", ascending=False)

gaps.style.format({"gap": "{:.1%}"}).hide(axis="index")

column,gap
credit_history,71.7%
affordability,31.9%
Property_Area,15.4%
Dependents,10.5%
Education,9.6%
Married,8.9%
Gender,7.8%
Self_Employed,0.5%


**Observation:** Ranking characteristics by the gap between their highest and lowest approval rate:
1. **Credit history** (71.7 points) is by far the strongest factor.
2. **Affordability** (31.9 points) matters, but only for the small group spending over 40% of
   income on repayments.
3. **Property area** (15.4 points): approval rates differ noticeably between semiurban, urban
   and rural applicants.

The remaining characteristics (dependents, education, marital status, gender, self-employment)
show gaps of 11 points or less. Some of those groups are small (for example 13 applicants with
unknown gender), so these differences may be due to chance and are not treated as clear effects.
These are associations in past decisions, not proof of cause.

### Q17. Credit history and affordability together
Approval rate (first table) and number of applicants (second table) for each combination.

In [27]:
loans_approved = loans.assign(approved=loans["Loan_Status"] == "Y")

# Approval rate in each combination
rate_table = pd.pivot_table(
    loans_approved,
    index="credit_history",
    columns="affordability",
    values="approved",
    aggfunc="mean",
    observed=False,
)

# Number of applicants in each combination
count_table = pd.pivot_table(
    loans_approved,
    index="credit_history",
    columns="affordability",
    values="Loan_ID",
    aggfunc="count",
    observed=False,
)

display(rate_table.style.format("{:.0%}", na_rep="–").background_gradient(cmap="Blues", vmin=0, vmax=1))
display(count_table)

affordability,Low (<15%),Medium (15–25%),High (25–40%),Very high (>40%)
credit_history,,,,
Bad,9%,8%,7%,0%
Good,77%,82%,77%,30%
Missing,67%,79%,67%,100%


affordability,Low (<15%),Medium (15–25%),High (25–40%),Very high (>40%)
credit_history,,,,
Bad,11,63,14,1
Good,102,307,56,10
Missing,12,24,12,2


**Observation:** Credit history and affordability work together:
- With a **bad** credit history, applicants were almost always rejected (0–9% approved),
  whatever their affordability.
- With a **good** credit history, approval stayed at about 80% as long as repayments took up to
  40% of income, then fell sharply to 30% above 40% (10 applicants). Affordability therefore
  matters even for applicants with a good history, but only at the extreme.
- **Missing** credit history behaves like a good one where groups are large enough (67–79%).
  Its 100% in the "very high" band is based on only 2 applicants and is not reliable.

This combination of credit history and affordability is the core of the hackathon idea: for
the same applicant, a higher interest rate raises the repayment and can move them into the
high-risk band.

## 6. Summary

**Data cleaning**
- All cleaning was done on a copy (`loans`), so the raw data stays unchanged and every step can be rerun.
- `Dependents` was converted from text to a whole number; `3+` became 3, meaning "3 or more".
- **Credit history:** the 50 missing values were kept as their own category, "Missing". Applicants
  without a credit history were approved at 74.0%, much closer to a good history (79.6%) than a bad
  one (7.9%), so filling them with "bad" would contradict the data and filling with "good" would
  invent information.
- **Other missing values:** `LoanAmount` was filled with the median, because loan amounts are
  right-skewed; `Loan_Amount_Term` (360), `Dependents` (0), `Married` and `Self_Employed` with the
  most common value, because one value clearly dominates or very few rows were affected.
  `Self_Employed` was checked first: approval rates were almost the same for all groups
  (68.3–71.9%). `Gender` was filled with "Unknown" rather than guessed.
- **Outliers:** 50 applicants (8.1%) have unusually high incomes (up to 81,000). They look genuine
  (mostly graduates requesting large loans) and were kept; medians are used to describe income.

**Affordability features**
- `total_income` = applicant + co-applicant income; its median (5,417) is 42% higher than the
  applicant's own income.
- `monthly_repayment` uses the standard loan repayment formula at an **assumed 9% interest rate**.
- `repayment_to_income` = monthly repayment ÷ total income; the median applicant would spend
  19.8% of household income on the loan.

**What drives approval**
1. **Credit history** is by far the strongest factor: 79.6% of applicants with a good history were
   approved, compared with 7.9% with a bad one. A bad history meant rejection in almost every case,
   whatever the affordability.
2. **Affordability** matters only at the extreme: approval stayed at about 70% for repayments up to
   25% of income, but fell to 38.5% above 40%. Even with a good credit history, approval dropped from
   about 80% to 30% when repayments exceeded 40% of income.
3. **Property area** is the third factor, with a 15.4-point gap in approval rates between areas.
- Other characteristics (dependents, education, marital status, gender, self-employment) showed
  no clear effect.

**Limitations**
- The dataset is small (614 applications), and some groups are very small, so their rates are uncertain.
- `Loan_Status` records **approval, not repayment**: the findings describe this lender's past
  decisions, not which borrowers actually repaid. A repayment model needs data with real outcomes.
- The interest rate (9%) and the units (loan amount in thousands, monthly incomes) are assumptions.
- The repayment formula does not fit a few very short-term loans (12–36 months).
- These are associations in past decisions, not proof of cause.